In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict

# Optional: make display nicer
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

DATA_DIR = Path("/content")   # put your 3 CSV files here in Colab
OUT_DIR = Path("/content/pharmgraph_pass1_artifacts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

NODES_FILE = DATA_DIR / "unified_graph_nodes.csv"
EDGES_FILE = DATA_DIR / "unified_graph_edges.csv"
SPLIT_FILE = DATA_DIR / "gene_drug_split.csv"

RANDOM_SEED = 42
NEG_RATIO = 1.0   # 1 negative per positive
rng = np.random.default_rng(RANDOM_SEED)

In [2]:
nodes = pd.read_csv(NODES_FILE)
edges = pd.read_csv(EDGES_FILE)
gene_drug_split = pd.read_csv(SPLIT_FILE)

print("nodes:", nodes.shape)
print("edges:", edges.shape)
print("gene_drug_split:", gene_drug_split.shape)

display(nodes.head())
display(edges.head())
display(gene_drug_split.head())

nodes: (27355, 2)
edges: (85593, 8)
gene_drug_split: (69006, 5)


,node_id,node_type
0,disease:pa443750,disease
1,disease:pa166128372,disease
2,disease:pa166124406,disease
3,disease:pa166123369,disease
4,disease:pa166114455,disease


,relation_type,source_id,target_id,source_raw,target_raw,source_type,target_type,split
0,gene_drug,gene:cyp2d6,drug:raclopride,cyp2d6,raclopride,gene,drug,train
1,gene_drug,gene:pparg,drug:chembl:chembl1833984,pparg,chembl:chembl1833984,gene,drug,train
2,gene_drug,gene:atad5,drug:chembl:chembl91609,atad5,chembl:chembl91609,gene,drug,test
3,gene_drug,gene:rgs4,"drug:3,4-dichloroisocoumarin",rgs4,"3,4-dichloroisocoumarin",gene,drug,train
4,gene_drug,gene:mapk1,drug:withaferin a,mapk1,withaferin a,gene,drug,train


,source_id,source_raw,target_id,target_raw,split
0,gene:hsd17b10,hsd17b10,drug:dephostatin,dephostatin,train
1,gene:hdac6,hdac6,drug:abexinostat,abexinostat,train
2,gene:f7,f7,drug:samotolisib,samotolisib,train
3,gene:kcna4,kcna4,drug:guanidine hydrochloride,guanidine hydrochloride,train
4,gene:vkorc1,vkorc1,drug:warfarin sodium,warfarin sodium,train


In [3]:
required_node_cols = {"node_id", "node_type"}
required_edge_cols = {
    "relation_type", "source_id", "target_id",
    "source_type", "target_type", "split"
}
required_split_cols = {"source_id", "target_id", "split"}

assert required_node_cols.issubset(nodes.columns), f"Missing node columns: {required_node_cols - set(nodes.columns)}"
assert required_edge_cols.issubset(edges.columns), f"Missing edge columns: {required_edge_cols - set(edges.columns)}"
assert required_split_cols.issubset(gene_drug_split.columns), f"Missing split columns: {required_split_cols - set(gene_drug_split.columns)}"

# Clean whitespace
for col in ["node_id", "node_type"]:
    nodes[col] = nodes[col].astype(str).str.strip()

for col in ["relation_type", "source_id", "target_id", "source_type", "target_type", "split"]:
    edges[col] = edges[col].astype(str).str.strip()

for col in ["source_id", "target_id", "split"]:
    gene_drug_split[col] = gene_drug_split[col].astype(str).str.strip()

# Drop exact duplicates if any
nodes = nodes.drop_duplicates(subset=["node_id"]).reset_index(drop=True)
edges = edges.drop_duplicates(subset=["relation_type", "source_id", "target_id"]).reset_index(drop=True)
gene_drug_split = gene_drug_split.drop_duplicates(subset=["source_id", "target_id"]).reset_index(drop=True)

print("After dedup:")
print("nodes:", nodes.shape)
print("edges:", edges.shape)
print("gene_drug_split:", gene_drug_split.shape)

After dedup:
nodes: (27355, 2)
edges: (85593, 8)
gene_drug_split: (69006, 5)


In [4]:
node_type_map = dict(zip(nodes["node_id"], nodes["node_type"]))

# Context edges = everything except gene_drug
context_edges = edges[edges["relation_type"] != "gene_drug"].copy()

# Target edges come from gene_drug_split.csv
target_pos = gene_drug_split.copy()

# Sanity: gene_drug split labels
assert set(target_pos["split"].unique()) <= {"train", "val", "test"}, "Unexpected split labels in gene_drug_split.csv"

train_pos = target_pos[target_pos["split"] == "train"].copy().reset_index(drop=True)
val_pos   = target_pos[target_pos["split"] == "val"].copy().reset_index(drop=True)
test_pos  = target_pos[target_pos["split"] == "test"].copy().reset_index(drop=True)

print("Initial positive split sizes")
print("train:", len(train_pos))
print("val:  ", len(val_pos))
print("test: ", len(test_pos))

Initial positive split sizes
train: 48304
val:   10350
test:  10352


In [9]:
def build_training_graph_edges(context_edges_df, train_pos_df):
    """
    Build the graph used for Node2Vec/GNN training:
    all context edges + only training gene_drug positive edges.
    """
    train_gene_drug_edges = train_pos_df.copy()
    train_gene_drug_edges["relation_type"] = "gene_drug"
    train_gene_drug_edges["source_type"] = train_gene_drug_edges["source_id"].map(node_type_map)
    train_gene_drug_edges["target_type"] = train_gene_drug_edges["target_id"].map(node_type_map)
    train_gene_drug_edges["split"] = "train"

    # Add missing columns if context has extra columns
    keep_cols = ["relation_type", "source_id", "target_id", "source_type", "target_type", "split"]
    train_gene_drug_edges = train_gene_drug_edges[keep_cols]

    context_keep = context_edges_df[keep_cols].copy()
    out = pd.concat([context_keep, train_gene_drug_edges], ignore_index=True)
    out = out.drop_duplicates(subset=["relation_type", "source_id", "target_id"]).reset_index(drop=True)
    return out


def get_training_graph_nodes(train_graph_edges_df):
    """Return all nodes appearing in the training graph edges."""
    return set(train_graph_edges_df["source_id"]).union(set(train_graph_edges_df["target_id"]))


def find_unsupported_pairs(eval_pos_df, training_graph_nodes):
    """
    Return eval rows where source_id or target_id is absent from the training graph.
    """
    mask = (
        ~eval_pos_df["source_id"].isin(training_graph_nodes) |
        ~eval_pos_df["target_id"].isin(training_graph_nodes)
    )
    return eval_pos_df[mask].copy()


def build_incident_map(df):
    """
    For each node, list incident positive gene_drug pairs as tuples,
    not dataframe row indices. This stays safe even if df is reindexed.
    """
    incident = defaultdict(list)
    for _, row in df.iterrows():
        pair = (row["source_id"], row["target_id"])
        incident[row["source_id"]].append(pair)
        incident[row["target_id"]].append(pair)
    return incident

In [8]:
def repair_split_for_transductive_node_support(train_pos_df, val_pos_df, test_pos_df, context_edges_df):
    """
    Ensure every node appearing in val/test positives is present in the training graph.

    Strategy:
      - Build provisional training graph
      - Find unsupported nodes in val/test
      - Move a minimal set of incident positive edges from val/test or test/val into train
      - Recompute until all val/test nodes are supported
    """
    train_pos_df = train_pos_df.copy().reset_index(drop=True)
    val_pos_df = val_pos_df.copy().reset_index(drop=True)
    test_pos_df = test_pos_df.copy().reset_index(drop=True)

    moved_rows = []

    def recompute_training_nodes():
        tg = build_training_graph_edges(context_edges_df, train_pos_df)
        return get_training_graph_nodes(tg)

    training_nodes = recompute_training_nodes()

    while True:
        unsupported_val = find_unsupported_pairs(val_pos_df, training_nodes)
        unsupported_test = find_unsupported_pairs(test_pos_df, training_nodes)

        unsupported_nodes = set()

        for df in [unsupported_val, unsupported_test]:
            for _, row in df.iterrows():
                if row["source_id"] not in training_nodes:
                    unsupported_nodes.add(row["source_id"])
                if row["target_id"] not in training_nodes:
                    unsupported_nodes.add(row["target_id"])

        if not unsupported_nodes:
            break

        moved_any = False

        # Rebuild incident maps fresh each round
        val_incident = build_incident_map(val_pos_df)
        test_incident = build_incident_map(test_pos_df)

        for node in sorted(unsupported_nodes):
            if node in training_nodes:
                continue

            candidate_source = None
            candidate_pair = None

            if node in val_incident and len(val_incident[node]) > 0:
                candidate_source = "val"
                candidate_pair = val_incident[node][0]
            elif node in test_incident and len(test_incident[node]) > 0:
                candidate_source = "test"
                candidate_pair = test_incident[node][0]

            if candidate_source is None or candidate_pair is None:
                continue

            s, t = candidate_pair

            if candidate_source == "val":
                match = (val_pos_df["source_id"] == s) & (val_pos_df["target_id"] == t)
                if match.sum() == 0:
                    continue
                row = val_pos_df.loc[match].iloc[0].copy()
                val_pos_df = val_pos_df.loc[~match].reset_index(drop=True)
            else:
                match = (test_pos_df["source_id"] == s) & (test_pos_df["target_id"] == t)
                if match.sum() == 0:
                    continue
                row = test_pos_df.loc[match].iloc[0].copy()
                test_pos_df = test_pos_df.loc[~match].reset_index(drop=True)

            row["split"] = "train"
            moved_rows.append({
                "source_id": row["source_id"],
                "target_id": row["target_id"],
                "moved_from": candidate_source,
                "reason": f"support node {node}"
            })

            train_pos_df = pd.concat([train_pos_df, pd.DataFrame([row])], ignore_index=True)
            train_pos_df = train_pos_df.drop_duplicates(subset=["source_id", "target_id"]).reset_index(drop=True)

            training_nodes = recompute_training_nodes()
            moved_any = True

        if not moved_any:
            remaining_val = find_unsupported_pairs(val_pos_df, training_nodes)
            remaining_test = find_unsupported_pairs(test_pos_df, training_nodes)
            raise RuntimeError(
                "Could not repair split: unsupported nodes remain but no incident positive edge could be moved.\n"
                f"Remaining unsupported val pairs: {len(remaining_val)}\n"
                f"Remaining unsupported test pairs: {len(remaining_test)}"
            )

    moved_df = pd.DataFrame(moved_rows)

    train_graph_edges_df = build_training_graph_edges(context_edges_df, train_pos_df)
    final_training_nodes = get_training_graph_nodes(train_graph_edges_df)

    assert len(find_unsupported_pairs(val_pos_df, final_training_nodes)) == 0
    assert len(find_unsupported_pairs(test_pos_df, final_training_nodes)) == 0

    return (
        train_pos_df.reset_index(drop=True),
        val_pos_df.reset_index(drop=True),
        test_pos_df.reset_index(drop=True),
        moved_df,
        train_graph_edges_df
    )

In [10]:
train_pos_fixed, val_pos_fixed, test_pos_fixed, moved_support_edges, train_graph_edges = \
    repair_split_for_transductive_node_support(train_pos, val_pos, test_pos, context_edges)

print("After repair")
print("train:", len(train_pos_fixed))
print("val:  ", len(val_pos_fixed))
print("test: ", len(test_pos_fixed))
print("moved support edges:", len(moved_support_edges))
print("train_graph_edges:", len(train_graph_edges))

display(moved_support_edges.head(20))

After repair
train: 51535
val:   8629
test:  8842
moved support edges: 3231
train_graph_edges: 68122


,source_id,target_id,moved_from,reason
0,gene:atf4,drug:&alpha;-conotoxin arib,val,support node drug:&alpha;-conotoxin arib
1,gene:serpinc1,drug:&alpha;-conotoxin mi,val,support node drug:&alpha;-conotoxin mi
2,gene:atf3,"drug:&alpha;-conotoxin mii [h9a, l15a]",val,"support node drug:&alpha;-conotoxin mii [h9a, l15a]"
3,gene:atf3,drug:&alpha;-conotoxin pia,val,support node drug:&alpha;-conotoxin pia
4,gene:aldh7a1p3,drug:&alpha;-dendrotoxin,val,support node drug:&alpha;-dendrotoxin
5,gene:tph2,drug:&alpha;-propyldopacetamide,val,support node drug:&alpha;-propyldopacetamide
6,gene:birc3,drug:&alpha;.&beta;-methylene-2-thio-udp,val,support node drug:&alpha;.&beta;-methylene-2-thio-udp
7,gene:angpt1,drug:&gamma;-msh,val,support node drug:&gamma;-msh
8,gene:atp5po,drug:&kappa;m-conotoxin riiik,val,support node drug:&kappa;m-conotoxin riiik
9,gene:nbeap1,drug:(+)-s20,val,support node drug:(+)-s20


In [11]:
training_nodes = get_training_graph_nodes(train_graph_edges)

unsupported_val_after = find_unsupported_pairs(val_pos_fixed, training_nodes)
unsupported_test_after = find_unsupported_pairs(test_pos_fixed, training_nodes)

print("Unsupported val after repair:", len(unsupported_val_after))
print("Unsupported test after repair:", len(unsupported_test_after))

assert len(unsupported_val_after) == 0
assert len(unsupported_test_after) == 0

# Ensure held-out positive edges are not in the training graph as gene_drug edges
train_gene_drug_pairs = set(map(tuple, train_pos_fixed[["source_id", "target_id"]].to_records(index=False)))
val_pairs = set(map(tuple, val_pos_fixed[["source_id", "target_id"]].to_records(index=False)))
test_pairs = set(map(tuple, test_pos_fixed[["source_id", "target_id"]].to_records(index=False)))

assert len(train_gene_drug_pairs & val_pairs) == 0
assert len(train_gene_drug_pairs & test_pairs) == 0
assert len(val_pairs & test_pairs) == 0

print("Positive split integrity checks passed.")

Unsupported val after repair: 0
Unsupported test after repair: 0
Positive split integrity checks passed.


In [12]:
all_genes = nodes.loc[nodes["node_type"] == "gene", "node_id"].drop_duplicates().tolist()
all_drugs = nodes.loc[nodes["node_type"] == "drug", "node_id"].drop_duplicates().tolist()

print("Genes:", len(all_genes))
print("Drugs:", len(all_drugs))

all_known_positive_pairs = set(
    map(tuple, pd.concat([train_pos_fixed, val_pos_fixed, test_pos_fixed], ignore_index=True)[["source_id", "target_id"]].to_records(index=False))
)

print("All known positive gene-drug pairs:", len(all_known_positive_pairs))
print("All possible gene-drug pairs:", len(all_genes) * len(all_drugs))
print("Negative candidate pool size:", len(all_genes) * len(all_drugs) - len(all_known_positive_pairs))

Genes: 5292
Drugs: 18481
All known positive gene-drug pairs: 69006
All possible gene-drug pairs: 97801452
Negative candidate pool size: 97732446


In [13]:
def sample_negative_pairs_for_split(
    pos_df,
    all_genes,
    all_drugs,
    known_positive_pairs,
    already_used_negative_pairs,
    ratio=1.0,
    rng=None
):
    """
    Sample gene-drug negative pairs for one split.

    Strategy:
    - Prefer genes and drugs that appear in this split's positives
    - Fall back to global genes/drugs if needed
    """
    if rng is None:
        rng = np.random.default_rng(42)

    n_needed = int(np.ceil(len(pos_df) * ratio))

    split_genes = pos_df["source_id"].drop_duplicates().tolist()
    split_drugs = pos_df["target_id"].drop_duplicates().tolist()

    sampled = set()
    attempts = 0
    max_attempts = max(200000, n_needed * 50)

    while len(sampled) < n_needed and attempts < max_attempts:
        attempts += 1

        # 85% of the time sample within split-local endpoints, else global
        if rng.random() < 0.85 and len(split_genes) > 0 and len(split_drugs) > 0:
            g = split_genes[rng.integers(0, len(split_genes))]
            d = split_drugs[rng.integers(0, len(split_drugs))]
        else:
            g = all_genes[rng.integers(0, len(all_genes))]
            d = all_drugs[rng.integers(0, len(all_drugs))]

        pair = (g, d)

        if pair in known_positive_pairs:
            continue
        if pair in already_used_negative_pairs:
            continue
        if pair in sampled:
            continue

        sampled.add(pair)

    if len(sampled) < n_needed:
        raise RuntimeError(f"Could only sample {len(sampled)} negatives out of {n_needed} needed.")

    neg_df = pd.DataFrame(sorted(sampled), columns=["source_id", "target_id"])
    neg_df["label"] = 0
    return neg_df

In [14]:
used_negative_pairs = set()

train_neg = sample_negative_pairs_for_split(
    train_pos_fixed,
    all_genes,
    all_drugs,
    known_positive_pairs=all_known_positive_pairs,
    already_used_negative_pairs=used_negative_pairs,
    ratio=NEG_RATIO,
    rng=rng
)
used_negative_pairs |= set(map(tuple, train_neg[["source_id", "target_id"]].to_records(index=False)))

val_neg = sample_negative_pairs_for_split(
    val_pos_fixed,
    all_genes,
    all_drugs,
    known_positive_pairs=all_known_positive_pairs,
    already_used_negative_pairs=used_negative_pairs,
    ratio=NEG_RATIO,
    rng=rng
)
used_negative_pairs |= set(map(tuple, val_neg[["source_id", "target_id"]].to_records(index=False)))

test_neg = sample_negative_pairs_for_split(
    test_pos_fixed,
    all_genes,
    all_drugs,
    known_positive_pairs=all_known_positive_pairs,
    already_used_negative_pairs=used_negative_pairs,
    ratio=NEG_RATIO,
    rng=rng
)
used_negative_pairs |= set(map(tuple, test_neg[["source_id", "target_id"]].to_records(index=False)))

print("Negative split sizes")
print("train_neg:", len(train_neg))
print("val_neg:  ", len(val_neg))
print("test_neg: ", len(test_neg))

Negative split sizes
train_neg: 51535
val_neg:   8629
test_neg:  8842


In [15]:
train_pos_labeled = train_pos_fixed.copy()
train_pos_labeled["label"] = 1

val_pos_labeled = val_pos_fixed.copy()
val_pos_labeled["label"] = 1

test_pos_labeled = test_pos_fixed.copy()
test_pos_labeled["label"] = 1

# Negatives must not overlap positives
known_positive_pairs = set(
    map(tuple, pd.concat([train_pos_fixed, val_pos_fixed, test_pos_fixed], ignore_index=True)[["source_id", "target_id"]].to_records(index=False))
)

for name, neg_df in [("train_neg", train_neg), ("val_neg", val_neg), ("test_neg", test_neg)]:
    neg_pairs = set(map(tuple, neg_df[["source_id", "target_id"]].to_records(index=False)))
    overlap = neg_pairs & known_positive_pairs
    assert len(overlap) == 0, f"{name} overlaps positives"

# Negatives must not overlap each other
train_neg_pairs = set(map(tuple, train_neg[["source_id", "target_id"]].to_records(index=False)))
val_neg_pairs   = set(map(tuple, val_neg[["source_id", "target_id"]].to_records(index=False)))
test_neg_pairs  = set(map(tuple, test_neg[["source_id", "target_id"]].to_records(index=False)))

assert len(train_neg_pairs & val_neg_pairs) == 0
assert len(train_neg_pairs & test_neg_pairs) == 0
assert len(val_neg_pairs & test_neg_pairs) == 0

print("Negative integrity checks passed.")

Negative integrity checks passed.


In [16]:
train_pairs = pd.concat([train_pos_labeled, train_neg], ignore_index=True)
val_pairs   = pd.concat([val_pos_labeled, val_neg], ignore_index=True)
test_pairs  = pd.concat([test_pos_labeled, test_neg], ignore_index=True)

# Shuffle
train_pairs = train_pairs.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
val_pairs   = val_pairs.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
test_pairs  = test_pairs.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print("Labeled pair tables")
print("train_pairs:", train_pairs.shape)
print("val_pairs:  ", val_pairs.shape)
print("test_pairs: ", test_pairs.shape)

display(train_pairs.head())

Labeled pair tables
train_pairs: (103070, 6)
val_pairs:   (17258, 6)
test_pairs:  (17684, 6)


,source_id,source_raw,target_id,target_raw,split,label
0,gene:pltp,NaN,"drug:[ser11](n-cnp,c-anp)pbnp2-15",NaN,NaN,0
1,gene:b2m,NaN,drug:coagulation factor viia (recombinant)-jncw,NaN,NaN,0
2,gene:csh2,NaN,drug:chembl:chembl1256984,NaN,NaN,0
3,gene:xk,xk,drug:bupivacaine,bupivacaine,train,1
4,gene:nedd4l,NaN,drug:tetradifon,NaN,NaN,0


In [17]:
# Fixed positive splits
train_pos_fixed.to_csv(OUT_DIR / "gene_drug_train_pos.csv", index=False)
val_pos_fixed.to_csv(OUT_DIR / "gene_drug_val_pos.csv", index=False)
test_pos_fixed.to_csv(OUT_DIR / "gene_drug_test_pos.csv", index=False)

# Negative splits
train_neg.to_csv(OUT_DIR / "gene_drug_train_neg.csv", index=False)
val_neg.to_csv(OUT_DIR / "gene_drug_val_neg.csv", index=False)
test_neg.to_csv(OUT_DIR / "gene_drug_test_neg.csv", index=False)

# Labeled pair tables
train_pairs.to_csv(OUT_DIR / "gene_drug_train_pairs_labeled.csv", index=False)
val_pairs.to_csv(OUT_DIR / "gene_drug_val_pairs_labeled.csv", index=False)
test_pairs.to_csv(OUT_DIR / "gene_drug_test_pairs_labeled.csv", index=False)

# Training graph edges for Node2Vec / GNN
train_graph_edges.to_csv(OUT_DIR / "train_graph_edges.csv", index=False)

# Support moves log
moved_support_edges.to_csv(OUT_DIR / "moved_support_edges.csv", index=False)

# Summary
summary = pd.DataFrame([
    {"artifact": "nodes", "count": len(nodes)},
    {"artifact": "context_edges", "count": len(context_edges)},
    {"artifact": "train_graph_edges", "count": len(train_graph_edges)},
    {"artifact": "train_pos_fixed", "count": len(train_pos_fixed)},
    {"artifact": "val_pos_fixed", "count": len(val_pos_fixed)},
    {"artifact": "test_pos_fixed", "count": len(test_pos_fixed)},
    {"artifact": "train_neg", "count": len(train_neg)},
    {"artifact": "val_neg", "count": len(val_neg)},
    {"artifact": "test_neg", "count": len(test_neg)},
    {"artifact": "moved_support_edges", "count": len(moved_support_edges)},
])
summary.to_csv(OUT_DIR / "pass1_split_summary.csv", index=False)

print(f"Saved artifacts to: {OUT_DIR}")
display(summary)

Saved artifacts to: /content/pharmgraph_pass1_artifacts


,artifact,count
0,nodes,27355
1,context_edges,16587
2,train_graph_edges,68122
3,train_pos_fixed,51535
4,val_pos_fixed,8629
5,test_pos_fixed,8842
6,train_neg,51535
7,val_neg,8629
8,test_neg,8842
9,moved_support_edges,3231


In [18]:
training_nodes = get_training_graph_nodes(train_graph_edges)

val_unsupported = find_unsupported_pairs(val_pos_fixed, training_nodes)
test_unsupported = find_unsupported_pairs(test_pos_fixed, training_nodes)

print("FINAL REPORT")
print("=" * 60)
print(f"Training graph nodes: {len(training_nodes):,}")
print(f"Training graph edges: {len(train_graph_edges):,}")
print(f"Moved support edges into train: {len(moved_support_edges):,}")
print(f"Unsupported val pairs after repair: {len(val_unsupported):,}")
print(f"Unsupported test pairs after repair: {len(test_unsupported):,}")
print(f"Train positives: {len(train_pos_fixed):,}")
print(f"Val positives: {len(val_pos_fixed):,}")
print(f"Test positives: {len(test_pos_fixed):,}")
print(f"Train negatives: {len(train_neg):,}")
print(f"Val negatives: {len(val_neg):,}")
print(f"Test negatives: {len(test_neg):,}")
print("=" * 60)

FINAL REPORT
Training graph nodes: 27,355
Training graph edges: 68,122
Moved support edges into train: 3,231
Unsupported val pairs after repair: 0
Unsupported test pairs after repair: 0
Train positives: 51,535
Val positives: 8,629
Test positives: 8,842
Train negatives: 51,535
Val negatives: 8,629
Test negatives: 8,842


In [19]:
import shutil
from google.colab import files

folder_path = "/content/pharmgraph_pass1_artifacts"
zip_base = "/content/pharmgraph_pass1_artifacts"

# Create zip file
shutil.make_archive(zip_base, "zip", folder_path)

# Download zip
files.download("/content/pharmgraph_pass1_artifacts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>